# ODI to Databricks Migration: WC_BADGE_DETAILS_D Conversion

## Source: `WC_BADGE_D` ODI Session
## Target: `workspace.prxbi_dw.wc_badge_details_d`

In [ ]:
%python
dbutils.widgets.text("ETL_JOB_TYPE", "FULL", "ETL Job Type (FULL/INCREMENTAL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "100", "Datasource ID")
dbutils.widgets.text("ETL_PROC_WID", "12345", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "999999", "ODI Session Number")
dbutils.widgets.text("etl_last_extract_time", "2024-01-01 00:00:00", "Last Extract Time for Incremental")
dbutils.widgets.text("etl_current_extract_time", "2024-12-31 23:59:59", "Current Extract Time for Incremental") # Default to end of year, will be overwritten by current_timestamp() in view

## 1. ETL Parameters and Views

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT CAST(getArgument('etl_last_extract_time') AS TIMESTAMP) AS value;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT current_timestamp() AS value;

In [ ]:
%sql
SELECT
  '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
  '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
  '${ETL_PROC_WID}' AS ETL_PROC_WID,
  '${ODI_SESS_NO}' AS ODI_SESS_NO,
  (SELECT value FROM v_etl_last_extract_time) AS etl_last_extract_time,
  (SELECT value FROM v_etl_current_extract_time) AS etl_current_extract_time;

## 2. Staging Table (`C$_WC_BADGE_D` -> `workspace.prxbi_dw.c_wc_badge_d`)

In [ ]:
%sql
-- SCEN_TASK_NO 30: Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_wc_badge_d;

In [ ]:
%sql
-- SCEN_TASK_NO 40: Create staging table
CREATE OR REPLACE TABLE workspace.prxbi_dw.c_wc_badge_d
(
    INTEGRATION_ID      STRING,
    BADGE_ID            STRING,
    BADGE_STATUS        BIGINT,
    CONTACT_EMAIL       STRING,
    ORG_NAME            STRING,
    CREATED_DATE        TIMESTAMP,
    LAST_UPDATED_DATE   TIMESTAMP,
    ETL_PROC_WID        BIGINT
)
USING DELTA;

## 3. Load Staging Table

In [ ]:
%sql
-- SCEN_TASK_NO 50: Load staging with incremental logic
INSERT INTO workspace.prxbi_dw.c_wc_badge_d
SELECT 
    b.INTEGRATION_ID,
    b.BADGE_ID,
    b.STATUS,
    b.CONTACT_EMAIL,
    b.ORGANISATION_NAME,
    b.CREATION_DATE,
    b.LAST_UPDATE_DATE,
    CAST('${ETL_PROC_WID}' AS BIGINT) AS ETL_PROC_WID
FROM workspace.prxbi_ts.src_badge_table b
WHERE b.LAST_UPDATE_DATE > (SELECT value FROM v_etl_last_extract_time)
  AND b.LAST_UPDATE_DATE <= (SELECT value FROM v_etl_current_extract_time);

In [ ]:
%sql
SELECT COUNT(1) AS staging_record_count
FROM workspace.prxbi_dw.c_wc_badge_d;

## 4. Target Table (`WC_BADGE_DETAILS_D` -> `workspace.prxbi_dw.wc_badge_details_d`)

In [ ]:
%sql
-- Create target table if it does not exist
CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.wc_badge_details_d
(
    ROW_WID             BIGINT GENERATED ALWAYS AS IDENTITY,
    INTEGRATION_ID      STRING,
    BADGE_ID            STRING,
    STATUS              BIGINT, -- Mapped from BADGE_STATUS
    CONTACT_EMAIL       STRING,
    ORG_NAME            STRING,
    CREATED_DATE        TIMESTAMP,
    LAST_UPDATED_DATE   TIMESTAMP,
    ETL_PROC_WID        BIGINT,
    W_INSERT_DT         TIMESTAMP,
    W_UPDATE_DT         TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO 100: Merge into target
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS tgt
USING (
    SELECT 
        INTEGRATION_ID,
        BADGE_ID,
        BADGE_STATUS,
        CONTACT_EMAIL,
        ORG_NAME,
        CREATED_DATE,
        LAST_UPDATED_DATE
    FROM workspace.prxbi_dw.c_wc_badge_d
) AS src
ON (tgt.INTEGRATION_ID = src.INTEGRATION_ID)
WHEN MATCHED THEN
    UPDATE SET
        tgt.STATUS          = src.BADGE_STATUS,
        tgt.CONTACT_EMAIL   = src.CONTACT_EMAIL,
        tgt.ORG_NAME        = src.ORG_NAME,
        tgt.W_UPDATE_DT     = current_timestamp()
WHEN NOT MATCHED THEN
    INSERT (
        BADGE_ID,
        STATUS,
        CONTACT_EMAIL,
        ORG_NAME,
        INTEGRATION_ID,
        W_INSERT_DT,
        W_UPDATE_DT,
        CREATED_DATE,
        LAST_UPDATED_DATE,
        ETL_PROC_WID
    ) VALUES (
        src.BADGE_ID,
        src.BADGE_STATUS,
        src.CONTACT_EMAIL,
        src.ORG_NAME,
        src.INTEGRATION_ID,
        current_timestamp(),
        current_timestamp(),
        src.CREATED_DATE,
        src.LAST_UPDATED_DATE,
        CAST('${ETL_PROC_WID}' AS BIGINT)
    );

## 5. Cleanup

In [ ]:
%sql
-- Cleanup staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_wc_badge_d;

## 6. Validation and Notes

In [ ]:
%sql
SELECT COUNT(1) AS total_records
FROM workspace.prxbi_dw.wc_badge_details_d;

**Conversion Notes:**
- Schema names have been mapped to `workspace.prxbi_dw` and `workspace.prxbi_ts` based on common patterns.
- Oracle `VARCHAR2`, `NUMBER`, `DATE` types converted to Spark `STRING`, `BIGINT`, `TIMESTAMP` respectively.
- `SYSDATE` converted to `current_timestamp()`.
- `TO_DATE('YYYY-MM-DD', 'YYYY-MM-DD')` in incremental filter replaced with dynamic values from `v_etl_last_extract_time` and `v_etl_current_extract_time` views.
- The target table `WC_BADGE_DETAILS_D` now includes `ROW_WID BIGINT GENERATED ALWAYS AS IDENTITY`.
- The `ROW_WID` column is explicitly excluded from the `MERGE` INSERT/UPDATE column lists, as per `GENERATED ALWAYS AS IDENTITY` rules.
- ETL parameters like `ETL_PROC_WID` are now passed via widgets.
- `C$_WC_BADGE_D` is created as a permanent Delta table, not a temporary view.
- The MERGE statement's INSERT clause was adjusted to include `CREATED_DATE`, `LAST_UPDATED_DATE`, and `ETL_PROC_WID` to ensure data from `C$` is fully integrated into the target, assuming these columns are present in the target dimension table. This aligns with preserving business logic where source data from `C$` is intended for the target.

**Manual Actions Required Post-Conversion:**
- Ensure `workspace.prxbi_ts.src_badge_table` exists and is accessible.
- Verify data types and nullability constraints on `workspace.prxbi_dw.wc_badge_details_d` after initial run.

In [ ]:
%python
# dbutils.widgets.remove("ETL_JOB_TYPE")
# dbutils.widgets.remove("DATASOURCE_NUM_ID")
# dbutils.widgets.remove("ETL_PROC_WID")
# dbutils.widgets.remove("ODI_SESS_NO")
# dbutils.widgets.remove("etl_last_extract_time")
# dbutils.widgets.remove("etl_current_extract_time")